<a href="https://colab.research.google.com/github/phoenixfin/aksantara-ocr/blob/main/notebooks/run_on_gpu.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Aksara OCR — benchmark run (Colab Pro, single session)

Built for a runtime where **nothing survives the session**. That single fact drives
the whole design:

1. **Runs are ordered seed-major** — every model is trained at seed 0 before any
   model is trained at seed 1. If the session dies at 60%, you have one complete
   seed across the whole matrix (a usable table) rather than three seeds across
   an arbitrary 60% of it (not a table).
2. **`--time-budget` stops cleanly between runs** instead of being killed
   mid-training, where an interrupted run leaves nothing at all.
3. **Every result is printed to cell output as it completes.** Saved notebook
   output is the one thing you reliably keep, so nothing lives only in memory.
4. **Results are zipped and downloaded to your machine at the end** — and you can
   run that cell at any point mid-experiment.

**Setup:** Runtime → Change runtime type → **L4 GPU** (or A100), and enable
**Background execution** so the run survives closing the tab.

In [1]:
import torch

assert torch.cuda.is_available(), \
    "No GPU. Runtime > Change runtime type > L4 GPU, then rerun this cell."

name = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"{name}  |  {vram:.1f} GB")
print(f"torch {torch.__version__}")

NVIDIA A100-SXM4-40GB  |  42.4 GB
torch 2.11.0+cu128


In [2]:
#@title Clone the repo and install dependencies
REPO_URL = "https://github.com/phoenixfin/aksantara-ocr.git"  #@param {type:"string"}

import os
from pathlib import Path

REPO = Path("/content/aksantara-ocr")
if REPO.exists():
    !cd {REPO} && git pull -q
else:
    !git clone -q {REPO_URL} {REPO}

os.chdir(REPO)
# torch/torchvision ship with Colab; reinstalling them is slow and occasionally
# pulls a version mismatched against the runtime's CUDA build.
!pip install -q timm pyyaml scikit-image tabulate
print(f"\nready: {Path.cwd()}")


ready: /content/aksantara-ocr


## Pull the dataset from Mendeley Data

Version 3 is already cleaned — 97,383 images, 889 classes, zero duplicates —
so the pipeline is just **fetch → cache → split**. No cleaning pass is needed
at runtime.

Nothing persists between sessions, so this runs every time. Fetching by DOI
keeps the notebook reproducible for anyone reading the paper: they run the same
cell and get byte-identical data.

In [3]:
#@title Fetch from Mendeley Data
DOI = "10.17632/vfj32bpjsf.3"  #@param {type:"string"}

# Look before downloading: confirms the DOI resolves and shows the size, so a
# typo costs a second rather than an 800 MB transfer.
!python scripts/00_fetch_mendeley.py --doi "{DOI}" --list-only

Dataset vfj32bpjsf v3

13 file(s), 808.3 MB total:

      230.0 MB  Bali.zip
       21.2 MB  Batak.zip
        2.4 MB  Bima.zip
       44.6 MB  Dunging-Iban.zip
      141.9 MB  Jawa.zip
        8.3 MB  Jawi.zip
       22.2 MB  Kawi.zip
       14.1 MB  Lampung.zip
       84.5 MB  Lontara.zip
       48.0 MB  Minangkabau.zip
       44.8 MB  Ogan.zip
       13.7 MB  Pallawa.zip
      132.7 MB  Sunda.zip


In [4]:
# Download and unpack the 13 per-script archives (~808 MB).
#
# Local disk, not Drive: reading ~100k small images over a mounted Drive is slow
# enough to become the bottleneck instead of the GPU.
!python scripts/00_fetch_mendeley.py --doi "{DOI}" --out /content/raw

RAW_ROOT = "/content/raw"
!find {RAW_ROOT} -maxdepth 1 -type d | head -15

Dataset vfj32bpjsf v3

13 file(s), 808.3 MB total:

      230.0 MB  Bali.zip
       21.2 MB  Batak.zip
        2.4 MB  Bima.zip
       44.6 MB  Dunging-Iban.zip
      141.9 MB  Jawa.zip
        8.3 MB  Jawi.zip
       22.2 MB  Kawi.zip
       14.1 MB  Lampung.zip
       84.5 MB  Lontara.zip
       48.0 MB  Minangkabau.zip
       44.8 MB  Ogan.zip
       13.7 MB  Pallawa.zip
      132.7 MB  Sunda.zip

  get : Bali.zip
       230.0 / 230.0 MB  (100.0%)
    extracted -> /content/raw

  get : Batak.zip
        21.2 / 21.2 MB  (100.0%)
    extracted -> /content/raw

  get : Bima.zip
         2.4 / 2.4 MB  (100.0%)
    extracted -> /content/raw

  get : Dunging-Iban.zip
        44.6 / 44.6 MB  (100.0%)
    extracted -> /content/raw

  get : Jawa.zip
       141.9 / 141.9 MB  (100.0%)
    extracted -> /content/raw

  get : Jawi.zip
         8.3 / 8.3 MB  (100.0%)
    extracted -> /content/raw

  get : Kawi.zip
        22.2 / 22.2 MB  (100.0%)
    extracted -> /content/raw

  get : Lampung.zip


In [5]:
#@title Fallbacks — uncomment one if needed

# Reproduce v3 from the raw v2 release. This is how v3 was produced; the rules
# file records every change with its evidence. Not needed for normal runs —
# v3 is already published in this state — but it lets a reviewer verify the
# cleaning independently.
# !python scripts/00_fetch_mendeley.py --doi "10.17632/vfj32bpjsf.2" --out /content/v2
# !python scripts/06_clean_dataset.py --data-root /content/v2 --out /content/raw \
#     --rules configs/cleaning_rules.yaml
# RAW_ROOT = "/content/raw"

# Upload a zip from your machine:
# from google.colab import files
# up = files.upload()
# !mkdir -p /content/raw && unzip -q -o "{list(up)[0]}" -d /content/raw
# RAW_ROOT = "/content/raw"

# Synthetic data, to rehearse the pipeline without the real corpus:
# !python scripts/make_synthetic_data.py --out /content/raw/synthetic
# RAW_ROOT = "/content/raw/synthetic"
pass

In [6]:
#@title Pre-resize into a decode-cheap cache — do not skip this
CACHE_SIZE = 224  #@param {type:"number"}

# Source images range up to 1500x1500. Measured on this corpus: decoding and
# resizing one costs ~6.7 ms per core, so Colab's dataloader workers sustain
# only ~300 img/s and the GPU idles. A 224px cache drops that to ~1 ms/img.
#
# Costs a few minutes once; saves hours across the matrix.
!python scripts/00b_build_cache.py \
    --data-root "{RAW_ROOT}" --out /content/data --size {CACHE_SIZE}

DATA_ROOT = "/content/data"

# Free the disk — the full-size tree is not needed again this session.
!rm -rf {RAW_ROOT}
!df -h /content | tail -1

Scanning /content/raw ...
97383 image(s); 0 already cached, 97383 to do.
  97383/97383  (1589 img/s, ETA 0.0 min)

Cached 97383 image(s) in 1.0 min -> /content/data

images: 97383 source -> 97383 cached
size  : 875 MB -> 782 MB

NOTE: 9 image(s) became byte-identical to another at 224px (they differ in the source).
      Downscaling merged them. They are a leakage path if split across train/test —
      pass --drop-duplicates to 01_prepare_data.py to remove them.

Next:
  python scripts/01_prepare_data.py --data-root /content/data --drop-duplicates
overlay         236G   48G  189G  21% /


In [7]:
#@title Scan and build the shared splits
ARTIFACTS = "/content/artifacts"

# stratified, not grouped: filenames carry no writer id, so writer-disjoint
# splitting is impossible. Reported accuracy is therefore an upper bound on
# generalization to unseen handwriting — state this in the paper.
#
# --drop-duplicates removes images that became byte-identical after the resize
# above. They are distinct in the source, but at cache resolution they are the
# same picture, and a merged pair split across train/test leaks the test set.
#
# Splits are computed once here and reused by every run, so cross-model
# comparisons cannot be affected by split luck.
!python scripts/01_prepare_data.py \
    --data-root "{DATA_ROOT}" \
    --out-dir "{ARTIFACTS}" \
    --split-strategy stratified \
    --drop-duplicates

Scanning /content/data ...

  images     : 97383
  scripts    : 13
  characters : 889 (script/character pairs)
  with writer id: 0/97383
  images per character: min=19 median=60 max=694

           Left in place, these can leak across the train/test boundary.
           Written to /content/artifacts/duplicates.csv for review.

Manifest -> /content/artifacts/manifest.csv

Dropped 9 duplicate image(s); 97374 remain.

Splitting (strategy=stratified, seed=42) ...
{
  "counts": {
    "train": 69552,
    "test": 13911,
    "val": 13911
  },
  "classes_per_split": {
    "train": 889,
    "val": 889,
    "test": 889
  },
  "total_classes": 889,
  "classes_absent_from_train": []
}

Splits -> /content/artifacts/splits.csv

Next: python scripts/02_run_matrix.py --config configs/full_benchmark.yaml


In [8]:
#@title Verify the data matches published v3
# Cheap insurance: catches a partial download, a wrong DOI, or a cache that
# silently lost files — in seconds, rather than after hours of training.
#
# manifest.csv is written before any filtering, so these three counts are
# invariant: they do not depend on cache resolution or --drop-duplicates.
import pandas as pd

manifest = pd.read_csv(f"{ARTIFACTS}/manifest.csv")
EXPECTED = {"images": 97383, "classes": 889, "scripts": 13}
actual = {
    "images": len(manifest),
    "classes": manifest["label"].nunique(),
    "scripts": manifest["script"].nunique(),
}

ok = True
for key, want in EXPECTED.items():
    got = actual[key]
    ok &= got == want
    print(f"  {key:8} {got:6}  expected {want:6}  {'OK' if got == want else 'MISMATCH'}")

if not ok:
    raise SystemExit(
        "Data does not match published v3.\n"
        "Fewer images than expected usually means the cache step dropped files — "
        "check its output.\n"
        "Re-run the fetch and cache cells. If you deliberately changed the "
        "dataset, update EXPECTED above."
    )

# Duplicates are NOT asserted: how many distinct images collapse into one
# depends on the cache resolution, so there is no single right number. They are
# reported here and removed from the splits by --drop-duplicates.
merged = len(manifest) - manifest["content_hash"].nunique()
splits = pd.read_csv(f"{ARTIFACTS}/splits.csv")
print(f"\n  {merged} image(s) became identical at cache resolution "
      f"(distinct in the source)")
print(f"  {len(splits)} image(s) in the splits after dropping them")
print("\nMatches published v3.")

  images    97383  expected  97383  OK
  classes     889  expected    889  OK
  scripts      13  expected     13  OK

  9 image(s) became identical at cache resolution (distinct in the source)
  97374 image(s) in the splits after dropping them

Matches published v3.


## Calibrate before committing the session

With no checkpointing, the matrix has to fit the session. Measure one run, then
multiply — don't guess. The next two cells give you a per-run cost and a count.

In [9]:
#@title Time a couple of short runs
!python scripts/02_run_matrix.py --config configs/smoke_test.yaml \
    --artifacts "{ARTIFACTS}" --results "{ARTIFACTS}/results/smoke" 2>&1 | grep -E "run:|acc=|mean"

[1/3] run: simplecnn_d2__unified__sz64__aug-medium__scratch__s0
    acc=0.0251  macro_f1=0.0010
[2/3] run: resnet18__unified__sz64__aug-medium__pt__s0
    acc=0.9308  macro_f1=0.8585
[3/3] run: resnet18__unified__sz64__aug-medium__scratch__s0
    acc=0.8076  macro_f1=0.6774
3 run(s) in 0.11h (mean 2.2min/run)


In [10]:
#@title Count the real matrix, and project the cost
CONFIG = "configs/layered.yaml"  #@param ["configs/layered.yaml", "configs/tier1_benchmark.yaml", "configs/tier1_ablation_aug.yaml", "configs/tier1_ablation_size.yaml", "configs/layered_benchmark.yaml", "configs/full_benchmark.yaml"]
MINUTES_PER_RUN = 8  #@param {type:"number"}
RESULTS = "/content/artifacts/results/main"

import re
import subprocess
from pathlib import Path

# Check the inputs this cell depends on before shelling out, so a skipped or
# failed earlier cell reports itself here rather than surfacing as a confusing
# parse error further down.
for required in ["manifest.csv", "splits.csv"]:
    if not (Path(ARTIFACTS) / required).exists():
        raise SystemExit(
            f"{Path(ARTIFACTS) / required} is missing.\n"
            "Run the 'Scan and build the shared splits' cell first (and check that "
            "the download / cache cells above it succeeded)."
        )

proc = subprocess.run(
    ["python", "scripts/02_run_matrix.py", "--config", CONFIG,
     "--artifacts", ARTIFACTS, "--results", RESULTS, "--dry-run"],
    capture_output=True, text=True,
)

# Report the real failure. Reading .stdout alone and parsing it blindly turns
# any error into "NoneType has no attribute 'group'", which says nothing about
# what actually went wrong.
if proc.returncode != 0:
    print(proc.stdout[-3000:])
    print(proc.stderr[-3000:])
    raise SystemExit(f"02_run_matrix.py failed (exit {proc.returncode}) — see output above.")

match = re.search(r"expands to (\d+)", proc.stdout)
if match is None:
    print(proc.stdout[-3000:])
    print(proc.stderr[-3000:])
    raise SystemExit("Could not find the run count in the output above.")

n = int(match.group(1))
hours = n * MINUTES_PER_RUN / 60
print(f"{n} runs x ~{MINUTES_PER_RUN}min = ~{hours:.1f}h")
print("seeds run in order, so a partial run still yields complete single-seed tables")
if hours > 10:
    print("\nLonger than one comfortable session. Either lower `seeds` in the config,"
          "\ndrop models, or set TIME_BUDGET below and accept a partial matrix.")

42 runs x ~8min = ~5.6h
seeds run in order, so a partial run still yields complete single-seed tables


In [11]:
#@title Run the selected matrix
TIME_BUDGET = 8.0  #@param {type:"number"}

# Runs whatever CONFIG you chose above. Set TIME_BUDGET a couple of hours under
# the session limit you expect: the runner predicts whether the next run fits
# and stops cleanly if it does not, so you always end on a completed run with
# results written and printed rather than being killed mid-training.
!python scripts/02_run_matrix.py --config "{CONFIG}" \
    --artifacts "{ARTIFACTS}" --results "{RESULTS}" \
    --time-budget {TIME_BUDGET}

Loaded 97374 images, 889 classes, 13 scripts.

Matrix expands to 42 experiments -> /content/artifacts/results/main
Device: cuda
[1/42] run: resnet18__script_id__sz64__aug-medium__pt__s0
    preloading 1.20 GB into memory
    acc=0.9991  macro_f1=0.9991
[2/42] run: resnet18__per_script__Bali__sz64__aug-medium__pt__s0
    preloading 0.17 GB into memory
preload 64px: 100% 9614/9614 [00:11<00:00, 839.50it/s]
preload 64px: 100% 1923/1923 [00:02<00:00, 840.54it/s]
preload 64px: 100% 1923/1923 [00:02<00:00, 844.86it/s]
    acc=0.9948  macro_f1=0.9941
[3/42] run: resnet18__per_script__Batak__sz64__aug-medium__pt__s0
    preloading 0.04 GB into memory
preload 64px: 100% 2524/2524 [00:02<00:00, 1025.93it/s]
preload 64px: 100% 505/505 [00:00<00:00, 1034.82it/s]
preload 64px: 100% 505/505 [00:00<00:00, 1042.66it/s]
    acc=0.9980  macro_f1=0.9980
[4/42] run: resnet18__per_script__Bima__sz64__aug-medium__pt__s0
    preloading 0.03 GB into memory
preload 64px: 100% 1537/1537 [00:01<00:00, 1434.94it/

In [12]:
#@title End-to-end evaluation of the layered classifier
# Requires both stages present at matching seeds — configs/layered.yaml runs
# script_id and per_script together, which is why they share one config.
#
# If CONFIG above was not layered.yaml, run it first:
# !python scripts/02_run_matrix.py --config configs/layered.yaml \
#     --artifacts "{ARTIFACTS}" --results "{RESULTS}" --time-budget 6

# Exactly computable, no extra inference: labels are script-qualified, so a
# misrouted image can never yield the correct label. Reports end-to-end
# accuracy decomposed into routing loss vs character loss, per script.
!python scripts/05_hierarchical_eval.py --results "{RESULTS}"


seed 0  (stage 1: resnet18)
  stage 1 — script accuracy      : 0.9991
  stage 2 — given correct routing: 0.9862
  END-TO-END                     : 0.9853
  accuracy lost to misrouting    : 0.0138

  per script (sorted by end-to-end):
 true_script    n  script_acc  end_to_end
        Jawi  458      0.9978      0.9476
        Jawa 1893      0.9974      0.9514
        Bima  308      1.0000      0.9838
       Sunda 4055      0.9990      0.9855
 Minangkabau  322      0.9969      0.9938
        Bali 1923      0.9995      0.9943
     Pallawa  560      1.0000      0.9946
     Lontara  914      1.0000      0.9956
        Ogan  525      0.9981      0.9962
Dunging-Iban  842      1.0000      0.9976
       Batak  505      1.0000      0.9980
        Kawi  892      1.0000      1.0000
     Lampung  714      1.0000      1.0000

seed 1  (stage 1: resnet18)
  stage 1 — script accuracy      : 0.9993
  stage 2 — given correct routing: 0.9853
  END-TO-END                     : 0.9845
  accuracy lost to mis

In [ ]:
#@title Ablations and the classical baseline (run if the budget allows)
# Both write into the same RESULTS dir, so the report aggregates everything.
# These justify the settings used above: if 64px matches 224px, that is worth
# stating with evidence rather than asserting.
for cfg in [
    "configs/tier1_ablation_size.yaml",
    "configs/tier1_ablation_aug.yaml",
]:
    print(f"\n{'=' * 70}\n{cfg}\n{'=' * 70}")
    !python scripts/02_run_matrix.py --config {cfg} \
        --artifacts "{ARTIFACTS}" --results "{RESULTS}" --time-budget 4

# CPU-only, so it costs no GPU time — and anchors the deep results against a
# non-deep baseline. If HOG+SVM lands close, that is itself a finding about
# how hard the dataset really is.
!python scripts/03_run_classical.py --artifacts "{ARTIFACTS}"


configs/tier1_ablation_size.yaml
Loaded 97374 images, 889 classes, 13 scripts.

Matrix expands to 18 experiments -> /content/artifacts/results/main
Device: cuda
[1/18] run: resnet18__unified__sz32__aug-medium__pt__s0
    preloading 0.30 GB into memory
preload 32px: 100% 69552/69552 [00:58<00:00, 1187.60it/s]
preload 32px: 100% 13911/13911 [00:11<00:00, 1184.29it/s]
preload 32px: 100% 13911/13911 [00:11<00:00, 1190.42it/s]
    acc=0.9651  macro_f1=0.9355
[2/18] run: resnet18__unified__sz48__aug-medium__pt__s0
    preloading 0.67 GB into memory
preload 48px: 100% 69552/69552 [01:02<00:00, 1111.45it/s]
preload 48px: 100% 13911/13911 [00:12<00:00, 1119.52it/s]
preload 48px: 100% 13911/13911 [00:12<00:00, 1117.09it/s]
    acc=0.9843  macro_f1=0.9698
[3/18] run: resnet18__unified__sz64__aug-medium__pt__s0
    preloading 1.20 GB into memory
epoch 8/40:  76% 821/1087 [00:23<00:07, 34.33it/s]

In [ ]:
#@title Print the tables (this output is saved with the notebook)
import pandas as pd
from pathlib import Path

pd.set_option("display.width", 220)
pd.set_option("display.max_rows", 400)
report = Path(RESULTS) / "report"

for name in ["main_benchmark", "ablation_augmentation", "ablation_image_size",
             "ablation_pretrained", "per_script"]:
    path = report / f"{name}.csv"
    if path.exists():
        print(f"\n{'=' * 70}\n{name}\n{'=' * 70}")
        print(pd.read_csv(path).to_string(index=False))

# Print the raw per-run table too — if the zip download fails, this output is
# still enough to rebuild every table by hand.
raw = report / "raw_results.csv"
if raw.exists():
    print(f"\n{'=' * 70}\nraw results (every run)\n{'=' * 70}")
    print(pd.read_csv(raw).to_string(index=False))

failures = Path(RESULTS) / "failures.jsonl"
if failures.exists():
    print(f"\nSome runs failed:\n{failures.read_text()[:3000]}")

In [ ]:
#@title Download everything to your machine
# Safe to run at any point, including mid-experiment from a second browser tab.
# Nothing here survives the session, so run it before you walk away.
from google.colab import files

!cd /content && zip -qr aksara_results.zip artifacts \
    -x "artifacts/results/*/*/test_predictions.npz"
!du -h /content/aksara_results.zip

# test_predictions.npz is excluded above because raw logits dominate the archive
# size. Drop the -x flag if you want them for post-hoc analysis.
files.download("/content/aksara_results.zip")

## If the session dies partway

You lose the run outputs, but not the experiment. Because ordering is
seed-major, whatever completed is a coherent slice: seed 0 across every model,
then seed 1, and so on. Concretely:

- **Saved notebook output** still contains every completed run's accuracy and
  macro-F1, printed as it finished.
- To continue, edit the config's `seeds:` to only the ones you still need
  (`seeds: [1, 2]`) and rerun. Splits are deterministic from the same seed, so
  a later session reproduces byte-identical splits and the results remain
  directly comparable.
- If you can tolerate mounting Drive for output, pointing `--artifacts` at
  `/content/drive/MyDrive/...` restores full resume — completed runs are then
  detected and skipped automatically.